# ASR demo

Set `DATASET_URL` (Google Drive link to a folder or .zip with `audio/` and optional `transcriptions/`), then Run all. Empty link = 10 utterances from LibriSpeech dev-clean.

In [ ]:
DATASET_URL = ""

In [ ]:
%cd /content
!rm -rf dla-asr-hw
!git clone -q https://github.com/wudzzlol/dla-asr-hw.git
%cd /content/dla-asr-hw
!git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt
# --no-deps: otherwise numpy gets downgraded to <2
!pip install -q --no-deps pyctcdecode pygtrie
!pip install -q hypothesis
# kenlm doesn't build on new python without regenerating the cython wrapper
!pip install -q "cython>=3"
!rm -rf /tmp/kenlm && git clone -q https://github.com/kpu/kenlm /tmp/kenlm
!cd /tmp/kenlm && cython --cplus -3 python/kenlm.pyx -o python/kenlm.cpp && pip install -q .
!python -c "import kenlm, pyctcdecode, numpy, torch; print('kenlm OK | numpy', numpy.__version__, '| torch', torch.__version__)"

In [ ]:
!python scripts/download_weights.py
!python scripts/download_lm.py
!ls -lh saved/ds2_full data/lm

In [ ]:
import glob, os, zipfile, gdown

DATA_ROOT = "/content/user_data"
os.makedirs(DATA_ROOT, exist_ok=True)

if DATASET_URL:
    if "/folders/" in DATASET_URL:
        gdown.download_folder(url=DATASET_URL, output=DATA_ROOT, quiet=False)
    else:
        zip_path = gdown.download(url=DATASET_URL, output=f"{DATA_ROOT}/data.zip", fuzzy=True, quiet=False)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(DATA_ROOT)
        os.remove(zip_path)
    search_root = DATA_ROOT
else:
    !python scripts/make_test_dir.py
    search_root = "data/test_custom_dir"

audio_dirs = [d for d in glob.glob(f"{search_root}/**/audio", recursive=True) if os.path.isdir(d)]
if os.path.isdir(f"{search_root}/audio"):
    audio_dirs.insert(0, f"{search_root}/audio")
assert audio_dirs, f"No 'audio' folder found inside {search_root}"
AUDIO_DIR = os.path.abspath(audio_dirs[0])
TRANS_DIR = os.path.join(os.path.dirname(AUDIO_DIR), "transcriptions")
HAS_TRANS = os.path.isdir(TRANS_DIR)
print("audio:", AUDIO_DIR, "|", len(os.listdir(AUDIO_DIR)), "files")
print("transcriptions:", TRANS_DIR if HAS_TRANS else "not found (only predictions will be saved)")

In [ ]:
trans_arg = f"datasets.test.transcription_dir={TRANS_DIR}" if HAS_TRANS else "metrics.inference=[]"
!python inference.py datasets=custom_dir datasets.test.audio_dir={AUDIO_DIR} {trans_arg} \
    metrics=lm \
    inferencer.from_pretrained=saved/ds2_full/model_best.pth \
    text_encoder.lm_model_path=data/lm/lm.arpa \
    text_encoder.vocab_path=data/lm/librispeech-vocab.txt \
    text_encoder.alpha=0.8 text_encoder.beta=3.0 \
    inferencer.decode_method=lm inferencer.beam_size=50 \
    inferencer.save_path=demo_preds

In [ ]:
pred_dirs = [d for d in glob.glob("**/demo_preds/test", recursive=True) if os.path.isdir(d)]
assert pred_dirs, "Predictions folder not found"
PRED_DIR = os.path.abspath(pred_dirs[0])
print(PRED_DIR, "|", len(os.listdir(PRED_DIR)), "files")

In [ ]:
if HAS_TRANS:
    !python calc_metrics.py --gt_dir {TRANS_DIR} --pred_dir {PRED_DIR}
else:
    print("No transcriptions -> nothing to compare against")

In [ ]:
for f in sorted(os.listdir(PRED_DIR))[:5]:
    pred = open(os.path.join(PRED_DIR, f), encoding="utf-8").read().strip()
    print(f"[{f}]")
    if HAS_TRANS and os.path.exists(os.path.join(TRANS_DIR, f)):
        print("  target:", open(os.path.join(TRANS_DIR, f), encoding="utf-8").read().strip().lower())
    print("  pred:  ", pred)